In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import time

from pathlib import Path
from comet_ml import Experiment

import sys
import os
sys.path.append(os.path.abspath(".."))

from ampute import generate_missing_data
from experiment_runner import load_dataset, run_single_experiment, log_experiment

np.random.seed(42)

In [ ]:
key = Path("../COMET_API_KEY.zshrc").read_text(encoding="utf-8").strip()
os.environ["COMET_API_KEY"] = key

In [ ]:
datasets = ["housing", "adult"]

# колонки, в которых будем создавать пропуски
columns_target = {
    "housing": [{"median_income": "total_rooms"}, 
                {"total_rooms": "population"}, 
                {"housing_median_age": "households"}],
    "adult": [{"occupation": "hours-per-week"},
              {"workclass": "hours-per-week"}, 
              {"hours-per-week": "age"}],
}

#  глобальный таргет
global_target = {
    "housing": "median_house_value",
    "adult": "income",
}

ratios = [0.05, 0.20, 0.50]

mechanisms = ["MCAR", "MAR"]

EXPERIMENT_CONFIG = {
    "Simple": [
        {"num_strategy": "mean"},
        {"num_strategy": "median"},
    ],
    "KNN numeric": [
        {"n_neighbors": 3},
        {"n_neighbors": 5},
        {"n_neighbors": 7},
    ],
    "KNN gower": [
        {"n_neighbors": 3},
        {"n_neighbors": 5},
        {"n_neighbors": 7},
    ],
    "KNN hybrid": [
        {"n_neighbors": 3},
        {"n_neighbors": 5},
        {"n_neighbors": 7},
    ],
    "MICE": [
        {"max_iter": 10},
        {"max_iter": 50},
    ],
    "MICE hybrid": [
        {"max_iter": 10},
        {"max_iter": 50},
    ],
    "MissForest": [
        {"n_estimators": 100},
        {"n_estimators": 200},
    ]
}
datasets = ["housing", "adult"]

# колонки, в которых будем создавать пропуски
columns_target = {
    "housing": [{"median_income": "total_rooms"}, 
                {"total_rooms": "population"}, 
                {"housing_median_age": "households"}],
    "adult": [{"occupation": "hours-per-week"},
              {"workclass": "hours-per-week"}, 
              {"hours-per-week": "age"}],
}

#  глобальный таргет
global_target = {
    "housing": "median_house_value",
    "adult": "income",
}

ratios = [0.05, 0.20, 0.50]

mechanisms = ["MCAR", "MAR"]

EXPERIMENT_CONFIG = {
    "Simple": [
        {"num_strategy": "mean"},
        {"num_strategy": "median"},
    ],
    "KNN numeric": [
        {"n_neighbors": 3},
        {"n_neighbors": 5},
        {"n_neighbors": 7},
    ],
    "KNN gower": [
        {"n_neighbors": 3},
        {"n_neighbors": 5},
        {"n_neighbors": 7},
    ],
    "KNN hybrid": [
        {"n_neighbors": 3},
        {"n_neighbors": 5},
        {"n_neighbors": 7},
    ],
    "MICE": [
        {"max_iter": 10},
        {"max_iter": 50},
    ],
    "MICE hybrid": [
        {"max_iter": 10},
        {"max_iter": 50},
    ],
    "MissForest": [
        {"n_estimators": 100},
        {"n_estimators": 200},
    ]
}


In [ ]:
for dataset_name in datasets:
    df = load_dataset(dataset_name)
    # удаляем глобальный таргет, чтобы случайно не подглядеть
    df = df.drop(columns=[global_target[dataset_name]])
    print(f"Dataset {dataset_name} loaded. Shape: {df.shape}")

    columns_with_missing = [list(d.keys())[0] for d in columns_target[dataset_name]]

    for mechanism in mechanisms:

        missing_data_cache = {}
        for ratio in ratios:
            df_incomplete, mask = generate_missing_data(
                data=df, 
                columns_config=columns_target[dataset_name],
                mechanism=mechanism, 
                ratio=ratio
            )
            missing_data_cache[ratio] = (df_incomplete, mask)
            print(f"Missing data generated for {dataset_name} | {mechanism} | Ratio: {ratio}")

        for method, param_list in EXPERIMENT_CONFIG.items():
            for params in param_list:
                params_str = "_".join([f"{k}={v}" for k, v in params.items()])

                # открываем эксперимент
                experiment = Experiment(
                    api_key=os.environ.get("COMET_API_KEY"),
                    project_name="missing-data-imputation",
                    workspace=None
                )
                experiment.set_name(f"{dataset_name} | {mechanism} | {method} ({params_str})")
                
                # логируем параметры 
                experiment.log_parameters({
                    "dataset": dataset_name,
                    "mechanism": mechanism,
                    "method": method,
                    **(params or {})
                })
                
                # прогоняем по проценту пропусков
                for ratio in ratios:
                    df_incomplete, mask = missing_data_cache[ratio]
                    
                    rmse, acc, time_taken = run_single_experiment(
                        df, 
                        df_incomplete, 
                        mask, 
                        method, 
                        params, 
                        columns_with_missing
                    )
                    
                    # логируем метрики
                    step = int(ratio * 100)
                    if rmse is not None:
                        experiment.log_metric("rmse", rmse, step=step)
                    if acc is not None:
                        experiment.log_metric("accuracy", acc, step=step)
                    if time_taken is not None:
                        experiment.log_metric("time", time_taken, step=step)
                    
                    rmse_str = f"{rmse:.4f}" if rmse is not None else "N/A"
                    acc_str = f"{acc:.4f}" if acc is not None else "N/A"

                print(f"Done: {dataset_name} | {mechanism} | {method} ({params_str})")
                experiment.end()

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.


Dataset housing loaded. Shape: (5000, 9)
Missing data generated for housing | MCAR | Ratio: 0.05
Missing data generated for housing | MCAR | Ratio: 0.2
Missing data generated for housing | MCAR | Ratio: 0.5


COMET INFO: Experiment is live on comet.com https://www.comet.com/kam2607kam/missing-data-imputation/bd2576025e8744e6b04d7c5406c7ba10

COMET INFO: The process of logging environment details (conda environment, git patch) is underway. Please be patient as this may take some time.
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : housing | MCAR | Simple (num_strategy=mean)
COMET INFO:     url                   : https://www.comet.com/kam2607kam/missing-data-imputation/bd2576025e8744e6b04d7c5406c7ba10
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     rmse [3] : (0.9997055557509777, 1.0002060071496477)
COMET INFO:     time [3] : (0.004123249993426725, 0.019771791994571686)
COMET INFO:   Others:


Done: housing | MCAR | Simple (num_strategy=mean)


COMET INFO: Please wait for metadata to finish uploading (timeout is 3600 seconds)
COMET INFO: Uploading 96 metrics, params and output messages
COMET INFO: Please wait for assets to finish uploading (timeout is 10800 seconds)
COMET INFO: Still uploading 1 file(s), remaining 1.37 MB/1.70 MB
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kam2607kam/missing-data-imputation/c43687ecf91246c790f4a58609cecd52

COMET INFO: The process of logging environment details (conda environment, git patch) is underway. Please be patient as this may take some time.
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Da

...

Done: adult | MAR | MICE hybrid (max_iter=50)


COMET INFO: Uploading 58 metrics, params and output messages
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kam2607kam/missing-data-imputation/6c8b754e8e344264a793cca04a50a35d

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : adult | MAR | MissForest (n_estimators=100)
COMET INFO:     url                   : https://www.comet.com/kam2607kam/missing-data-imputation/6c8b754e8e344264a793cca04a50a35d
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     accuracy [3] : (0.482, 0.532)
COMET INFO:     rmse [3]   

Done: adult | MAR | MissForest (n_estimators=100)


COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kam2607kam/missing-data-imputation/97a341c3ad9a402981954ae70d870c2e

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : adult | MAR | MissForest (n_estimators=200)
COMET INFO:     url                   : https://www.comet.com/kam2607kam/missing-data-imputation/97a341c3ad9a402981954ae70d870c2e
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     accuracy [3] : (0.484, 0.534)
COMET INFO:     rmse [3]     : (0.9610887099440625, 1.0417489423273905)
COMET INFO:     

Done: adult | MAR | MissForest (n_estimators=200)
